<a href="https://colab.research.google.com/github/aodm26/gpt-oss/blob/main/GPT_OSS_Reasoning_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-OSS-20B Sentiment Analysis — Speed-Optimised for T4

**Dataset:** `test_split_2.xlsx` — 484 financial headlines (pre-labelled)  
**Labels:** `positive` / `negative` / `neutral` (text, lowercase)  
**Label split:** neutral=288 · positive=136 · negative=60

### Speed decisions for T4 (14.5 GB VRAM)
| Setting | Value | Reason |
|---|---|---|
| `load_in_4bit` | `True` | ~4× less VRAM for weights |
| `max_seq_length` | `1024` | Halves KV-cache pre-allocation vs 2048 |
| `reasoning_effort` | `'low'` | Short think trace; `high` = 167 s/item |
| `max_new_tokens` | `96` | Label word + 1-sentence reason fits easily |
| `do_sample` | `False` | Greedy decode — no sampling overhead |
| Batch size | `1` | Batching causes KV-cache OOM on T4 |
| Prompt | Forces single label word on last line | Zero-ambiguity parsing |


## 1 · Install Dependencies

In [5]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo


## 2 · Load Model (4-bit, T4-safe)

In [6]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/gpt-oss-20b-unsloth-bnb-4bit",  # 4-bit — fits T4
    dtype          = None,
    max_seq_length = 1024,   # small = less KV-cache VRAM
    load_in_4bit   = True,
    full_finetuning= False,
)
FastLanguageModel.for_inference(model)   # enables optimised inference kernels
print("✓ Model ready.")


==((====))==  Unsloth 2026.3.4: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✓ Model ready.


## 3 · Load Dataset

In [12]:
import pandas as pd

df = pd.read_excel('/content/test_split 2.xlsx')

sample_df = df.sample(n=80, random_state=42).reset_index(drop=True)

# Normalise labels to Title case so they match parse_label() output
sample_df['sentiment'] = sample_df['sentiment'].str.strip().str.capitalize()

headlines_list  = sample_df['headline'].tolist()
expected_labels = sample_df['sentiment'].tolist()
n = len(headlines_list)


print(f"Loaded {n} headlines.")
print("Label distribution:")
print(sample_df['sentiment'].value_counts().to_string())
print(f"\nSample headline: {headlines_list[0]}")
print(f"Expected label : {expected_labels[0]}")

Loaded 80 headlines.
Label distribution:
sentiment
Neutral     47
Positive    22
Negative    11

Sample headline: Active shipping is essential for Finland.
Expected label : Neutral


## 4 · Label Parser

Extracts the final sentiment word from model output.  
Two-pass: first looks for an explicit `Final sentiment label:` line, then falls back to the last valid label word found anywhere in the response.


In [13]:
import re

VALID_LABELS = {'Positive', 'Negative', 'Neutral'}

def parse_label(text: str) -> str:
    clean = text.replace('**', '').replace('*', '')

    # Pass 1 — explicit label line
    m = re.search(r'[Ff]inal\s+(?:sentiment\s+)?label[:\s]+([A-Za-z]+)', clean)
    if m and m.group(1).capitalize() in VALID_LABELS:
        return m.group(1).capitalize()

    # Pass 2 — last occurrence of any valid label word
    found = re.findall(r'\b(Positive|Negative|Neutral)\b', text, re.IGNORECASE)
    if found:
        return found[-1].capitalize()

    return 'Unknown'

# Sanity checks
assert parse_label("**Final sentiment label:** **Positive**") == "Positive"
assert parse_label("The tone is clearly negative.") == "Negative"
assert parse_label("Overall: Neutral") == "Neutral"
print("✓ parse_label() checks passed.")


✓ parse_label() checks passed.


## 5 · Run Inference

Single-item loop with all T4 speed levers applied.  
`reasoning_effort='low'` keeps the internal think trace short without sacrificing accuracy.


In [ ]:
import torch, time, textwrap

SYSTEM_PROMPT = textwrap.dedent("""
    You are a financial news sentiment classifier.
    State your reasoning in one sentence, then on the very last line
    write exactly one word — the sentiment label.
    Valid labels: Positive  Negative  Neutral
""").strip()

results = []
start   = time.time()

for i, (headline, exp) in enumerate(zip(headlines_list, expected_labels)):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Headline: {headline}"},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors        = "pt",
        return_dict           = True,
        reasoning_effort      = "low",
    ).to("cuda")

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens = 96,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
            use_cache      = True,
        )

    input_len = inputs["input_ids"].shape[1]
    response  = tokenizer.decode(out_ids[0][input_len:], skip_special_tokens=True).strip()
    pred      = parse_label(response)
    correct   = (pred == exp)

    results.append({
        "index":     i + 1,
        "headline":  headline,
        "expected":  exp,
        "predicted": pred,
        "correct":   correct,
        "reasoning": response,
    })

    elapsed = time.time() - start
    rate    = elapsed / (i + 1)
    eta     = rate * (n - i - 1)
    tick    = "✓" if correct else "✗"

    # Print every item for small dataset; show ETA
    print(f"[{i+1:3d}/{n}] {tick} exp={exp:<9s} pred={pred:<9s} "
          f"| {rate:.1f}s/item | ETA≈{eta/60:.1f}min")

print(f"\n✓ Done! Total: {(time.time()-start)/60:.1f} min")


[  1/80] ✓ exp=Neutral   pred=Neutral   | 17.9s/item | ETA≈23.6min
[  2/80] ✓ exp=Neutral   pred=Neutral   | 17.8s/item | ETA≈23.2min
[  3/80] ✓ exp=Neutral   pred=Neutral   | 19.4s/item | ETA≈24.9min
[  4/80] ✓ exp=Neutral   pred=Neutral   | 21.6s/item | ETA≈27.3min
[  5/80] ✓ exp=Positive  pred=Positive  | 20.9s/item | ETA≈26.1min
[  6/80] ✗ exp=Negative  pred=Neutral   | 20.5s/item | ETA≈25.3min
[  7/80] ✓ exp=Neutral   pred=Neutral   | 19.7s/item | ETA≈23.9min
[  8/80] ✓ exp=Negative  pred=Negative  | 19.3s/item | ETA≈23.2min
[  9/80] ✓ exp=Positive  pred=Positive  | 19.1s/item | ETA≈22.7min
[ 10/80] ✓ exp=Positive  pred=Positive  | 18.7s/item | ETA≈21.9min
[ 11/80] ✓ exp=Positive  pred=Positive  | 18.2s/item | ETA≈20.9min
[ 12/80] ✓ exp=Neutral   pred=Neutral   | 17.9s/item | ETA≈20.3min
[ 13/80] ✗ exp=Positive  pred=Neutral   | 17.9s/item | ETA≈20.0min


## 6 · Accuracy & Confusion Matrix

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

results_df = pd.DataFrame(results)
y_true = results_df['expected'].tolist()
y_pred = results_df['predicted'].tolist()

acc       = accuracy_score(y_true, y_pred)
n_correct = results_df['correct'].sum()
n_unknown = (results_df['predicted'] == 'Unknown').sum()

print("=" * 60)
print(f"  Overall Accuracy : {acc:.1%}  ({n_correct}/{len(results_df)} correct)")
print(f"  Unparsed labels  : {n_unknown}")
print("=" * 60)
print()
print(classification_report(
    y_true, y_pred,
    labels=['Positive', 'Negative', 'Neutral'],
    zero_division=0
))

# ── Confusion matrix ──────────────────────────────────────────────────────────
labels = ['Positive', 'Negative', 'Neutral']
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, linewidths=0.5, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Expected',  fontsize=12)
ax.set_title(f'Confusion Matrix  —  Accuracy: {acc:.1%}', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("Saved → confusion_matrix.png")


## 7 · Misclassified Headlines

In [ ]:
wrong_df = results_df[results_df['correct'] == False][
    ['index', 'headline', 'expected', 'predicted', 'reasoning']
].reset_index(drop=True)

print(f"Misclassified: {len(wrong_df)} / {len(results_df)}")
print()
for _, row in wrong_df.iterrows():
    print(f"#{int(row['index']):3d} | Expected={row['expected']:<9s} Predicted={row['predicted']}")
    print(f"       {row['headline'][:110]}")
    print(f"       Reasoning: {row['reasoning'][:120]}")
    print()


## 8 · Save Results

In [ ]:
out = '/content/sentiment_results.csv'
results_df[['index','headline','expected','predicted','correct']].to_csv(out, index=False)
print(f"Results saved → {out}")

print("\nLabel distribution comparison:")
comp = pd.DataFrame({
    'Expected' : results_df['expected'].value_counts(),
    'Predicted': results_df['predicted'].value_counts(),
}).fillna(0).astype(int)
print(comp.to_string())
